# ABS Mesh Block and SA1 feature extraction for the model sites

This notebook extracts one source-specific feature table for every surveyed site in `data/processed/model_dataset.xlsx`. It uses each site's WGS84 coordinate to identify the containing 2021 ABS Mesh Block (MB) and Statistical Area Level 1 (SA1), then joins the official WA Mesh Block category, area, dwelling count, and usual-resident population.

## What this data source is

This workflow combines two official Australian Bureau of Statistics (ABS) components from the 2021 Census geography:

- The **ASGS Edition 3 (2021) Mesh Block spatial service** supplies the MB and SA1 containing each project coordinate. A Mesh Block is the smallest geographic unit in the ASGS; SA1 is the next Census geography assembled from whole Mesh Blocks.
- The **Census of Population and Housing: Mesh Block Counts, 2021** workbook supplies the dominant MB category, official area, dwelling count, and usual-resident population for Western Australia.

### API used

The notebook uses the official **ABS ArcGIS REST API** for the ASGS 2021 Mesh Block layer:

`https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/MB/MapServer/0/query`

No API key is required. For each site, the notebook sends longitude and latitude as a WGS84 point (`EPSG:4326`) and performs a point–polygon intersection query. It requests only `mb_code_2021` and `sa1_code_2021`, without returning polygon geometry. The API identifies the containing geography; it does **not** return dwelling or population counts. Those attributes are joined afterwards from the local official ABS workbook through `mb_code_2021`.

The spatial service answers **which official area contains this point**. The counts workbook answers **what basic Census context is recorded for that containing Mesh Block**. Together they provide a small-area land-use and residential-context description for each survey site.

## What information we can obtain

For each `site_id`, this source can identify its 2021 MB and SA1, the dominant land-use category of the MB, the MB's official area, its total dwellings, and its usual-resident population. Because Mesh Blocks have unequal areas, the notebook also calculates dwellings per square kilometre and usual residents per square kilometre.

It cannot directly provide pedestrian activity, current population, socioeconomic characteristics, or the number of residents and destinations around the site within a chosen walking buffer. Those questions require other datasets or a separate neighbourhood-buffer method.

## Final output fields and their meaning

The output contains identifiers and QA fields as well as candidate model features. They must not all be passed directly into a model.

| Output field | Type of field | Plain meaning |
|---|---|---|
| `site_id` | Project join key | Identifies the survey site and links this table back to `Model_Data`. |
| `abs_match_status` | QA field | States whether the coordinate produced one usable ABS Mesh Block match. |
| `abs_error_message` | QA field | Explains a failed, missing, or ambiguous match. |
| `abs_mb_code_2021` | Geographic join key | Official code of the containing 2021 Mesh Block. Keep for traceability; do not use it as a continuous numeric predictor. |
| `abs_sa1_code_2021` | Geographic join key | Official code of the containing 2021 SA1. Keep for future SA1-level joins; do not use it as a continuous numeric predictor. |
| `abs_mb_category_2021` | Candidate categorical feature | Dominant MB land-use context, for example Residential, Commercial, Transport, Parkland, or Industrial. It is not a land-use-mix score. |
| `abs_mb_area_sqkm_2021` | Candidate numeric/context field | Official area of the containing MB in square kilometres. It describes polygon size and is also the density denominator. |
| `abs_mb_dwellings_2021` | Candidate numeric feature | Total dwellings recorded in the containing MB. |
| `abs_mb_persons_usual_resident_2021` | Candidate numeric feature | People whose usual residence is in the containing MB; this is not the number of pedestrians observed there. |
| `abs_mb_dwellings_per_sqkm_2021` | Derived candidate numeric feature | Dwelling count divided by MB area, used to compare residential intensity across unequal MB sizes. |
| `abs_mb_persons_per_sqkm_2021` | Derived candidate numeric feature | Usual-resident count divided by MB area, used to compare resident density across unequal MB sizes. |

The six candidate model fields are therefore MB category, MB area, raw dwelling count, raw usual-resident count, dwelling density, and usual-resident density. They are extracted for evaluation, not automatically approved for the final model. With a small modelling sample, correlated raw counts and densities should be compared during feature selection rather than all being forced into the same model.

The data roles are deliberately separated:

- `model_dataset.xlsx` remains the stable foundation table for site identifiers, coordinates, targets, and project QA. It is read but never modified by this notebook.
- `Mesh Block Counts, 2021.xlsx` is the local official ABS count table.
- The ABS ArcGIS REST service supplies the spatial lookup from each point to its 2021 MB and SA1 codes.
- `haoqi-01-abs-mb-sa1-features.xlsx` is a derived, source-specific feature table with one row per `site_id`. It can later be joined to the foundation table in modelling code.

All sites are queried, regardless of the target or QA fields, because feature extraction depends only on location. Any training filter belongs in the modelling stage. The output does not copy targets, road attributes, or other foundation-table columns.

Official references used to interpret the data:

- [2021 Census Mesh Block counts](https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021)
- [ASGS Edition 3 Mesh Blocks](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/main-structure-and-greater-capital-city-statistical-areas/mesh-blocks)
- [ASGS Edition 3 Statistical Area Level 1](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/main-structure-and-greater-capital-city-statistical-areas/statistical-area-level-1)
- [ASGS Edition 3 data services and APIs](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/data-services-and-apis)

Important scientific limitation: these variables describe the **Mesh Block containing the survey coordinate**. They are not counts or densities within a walking-distance buffer around the site. A road point can lie in a Transport, Parkland, Commercial, or other zero-population MB even when residents live nearby.

In [1]:
from pathlib import Path
import json

import pandas as pd
import requests
from IPython.display import display


def find_project_root() -> Path:
    """Find the project repository root on the host or in the container."""
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").is_file() and (path / "data").is_dir():
            return path
    raise FileNotFoundError(
        f"Project root not found above {current}. Expected requirements.txt and data/."
    )


PROJECT_ROOT = find_project_root()
MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
ABS_COUNTS_PATH = (
    PROJECT_ROOT / "data" / "external_raw" / "Mesh Block Counts, 2021.xlsx"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "outputs" / "02_imagery_features" / "abs-mesh-block-sa1"
)
FEATURE_PATH = OUTPUT_DIR / "haoqi-01-abs-mb-sa1-features.xlsx"

MB_QUERY_URL = (
    "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/MB/MapServer/0/query"
)

for required_file in [MODEL_DATASET_PATH, ABS_COUNTS_PATH]:
    if not required_file.is_file():
        raise FileNotFoundError(f"Required input file not found: {required_file}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Model input  :", MODEL_DATASET_PATH.relative_to(PROJECT_ROOT))
print("ABS counts   :", ABS_COUNTS_PATH.relative_to(PROJECT_ROOT))
print("Feature file :", FEATURE_PATH.relative_to(PROJECT_ROOT))

Project root : /workspace
Model input  : data/processed/model_dataset.xlsx
ABS counts   : data/external_raw/Mesh Block Counts, 2021.xlsx
Feature file : outputs/02_imagery_features/abs-mesh-block-sa1/haoqi-01-abs-mb-sa1-features.xlsx


## 1. Read the model-site input

The `Model_Data` sheet is the authoritative site list. Only four columns are needed here:

| Input column | Definition and use in this notebook |
|---|---|
| `site_id` | Stable survey-site primary key. It is preserved as text and becomes the only join key for the feature output. |
| `latitude` | WGS84 latitude in decimal degrees; sent to the ABS spatial service. |
| `longitude` | WGS84 longitude in decimal degrees; sent to the ABS spatial service. |
| `coordinate_crs` | Coordinate reference system recorded by the project. The spatial request below requires `EPSG:4326`. |

The checks below are limited to conditions needed for a defensible spatial join: required columns must exist, `site_id` must be complete and unique, coordinates must be numeric and geographically valid, and the CRS must be WGS84. Coordinates are used during processing but are not duplicated in the final feature table.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)

site_columns = ["site_id", "latitude", "longitude", "coordinate_crs"]
missing_columns = set(site_columns) - set(model_data.columns)
if missing_columns:
    raise ValueError(f"Model_Data is missing required columns: {sorted(missing_columns)}")

sites = model_data[site_columns].copy()
sites["site_id"] = sites["site_id"].str.strip()
sites["coordinate_crs"] = sites["coordinate_crs"].str.strip().str.upper()
sites["latitude"] = pd.to_numeric(sites["latitude"], errors="coerce")
sites["longitude"] = pd.to_numeric(sites["longitude"], errors="coerce")

if sites["site_id"].isna().any() or sites["site_id"].eq("").any():
    raise ValueError("site_id contains missing or blank values.")
if sites["site_id"].duplicated().any():
    duplicates = sites.loc[sites["site_id"].duplicated(), "site_id"].tolist()
    raise ValueError(f"site_id must be unique. Duplicates: {duplicates}")

invalid_coordinates = (
    sites["latitude"].isna()
    | sites["longitude"].isna()
    | ~sites["latitude"].between(-90, 90)
    | ~sites["longitude"].between(-180, 180)
)
if invalid_coordinates.any():
    bad_sites = sites.loc[invalid_coordinates, "site_id"].tolist()
    raise ValueError(f"Invalid or missing coordinates for: {bad_sites}")

unsupported_crs = sites.loc[sites["coordinate_crs"].ne("EPSG:4326"), "site_id"]
if not unsupported_crs.empty:
    raise ValueError(f"Coordinates must use EPSG:4326. Check: {unsupported_crs.tolist()}")

print(f"Model sites ready for feature extraction: {len(sites):,}")
display(sites.head())

Model sites ready for feature extraction: 49


,site_id,latitude,longitude,coordinate_crs
0,250303-01,-32.866282,115.705163,EPSG:4326
1,250303-02,-33.359642,115.678041,EPSG:4326
2,250303-03,-33.360624,116.154738,EPSG:4326
3,250303-04,-34.238746,116.149491,EPSG:4326
4,250303-05,-32.628918,115.874612,EPSG:4326


## 2. Read the official WA Mesh Block attributes

The workbook `Contents` sheet identifies **Table 5** as Western Australia. Its `Explanatory Notes` sheet defines the source fields. The selected fields and their roles are:

| ABS source field | Official meaning | Role in the feature table |
|---|---|---|
| `MB_CODE_2021` | Eleven-digit 2021 Mesh Block identifier. The first digit is the State or Territory code. | Geographic join and traceability field; do not treat it as a numeric predictor. |
| `MB_CATEGORY_NAME_2021` | Dominant land-use category assigned during MB design, such as Residential, Commercial, Industrial, Parkland, Education, Hospital/Medical, Transport, Primary Production, Water, or Other. | Candidate categorical context feature. It is not a land-use-mix measure. |
| `AREA_ALBERS_SQKM` | Mesh Block area in square kilometres, calculated using the Albers Equal Area Conic projection. | Geographic scale and denominator for the density calculations. |
| `Dwelling` | Total dwellings. ABS counts structures intended and suitable for people to live in on Census Night, including occupied dwellings and most unoccupied private dwellings. | Raw dwelling count and numerator for dwelling density. |
| `Person` | Persons usually resident, counted at the place where they usually live rather than necessarily where they were on Census Night. | Raw usual-resident count and numerator for resident density. It is not observed pedestrian activity. |
| `State` | One-digit State or Territory code; `5` represents Western Australia. | Used to verify the WA rows, then excluded from the feature output. |

The raw counts, area, and category are retained so the density values remain auditable. Density improves comparability across unequal MB areas, but it may be extreme for very small blocks and must be interpreted together with area and category.

In [3]:
wa_counts = pd.read_excel(
    ABS_COUNTS_PATH,
    sheet_name="Table 5",
    header=6,
    dtype={"MB_CODE_2021": "string"},
)

abs_columns = [
    "MB_CODE_2021",
    "MB_CATEGORY_NAME_2021",
    "AREA_ALBERS_SQKM",
    "Dwelling",
    "Person",
    "State",
]
missing_columns = set(abs_columns) - set(wa_counts.columns)
if missing_columns:
    raise ValueError(f"ABS Table 5 is missing required fields: {sorted(missing_columns)}")

wa_counts = wa_counts[abs_columns].copy()
wa_counts["MB_CODE_2021"] = wa_counts["MB_CODE_2021"].str.strip()
wa_counts["State"] = pd.to_numeric(wa_counts["State"], errors="coerce")
wa_counts = wa_counts.loc[
    wa_counts["State"].eq(5)
    & wa_counts["MB_CODE_2021"].str.fullmatch(r"5\d{10}", na=False)
].copy()

for column in ["AREA_ALBERS_SQKM", "Dwelling", "Person"]:
    wa_counts[column] = pd.to_numeric(wa_counts[column], errors="coerce")

if wa_counts["MB_CODE_2021"].duplicated().any():
    raise ValueError("MB_CODE_2021 must be unique before joining.")

wa_counts = wa_counts.rename(
    columns={
        "MB_CODE_2021": "abs_mb_code_2021",
        "MB_CATEGORY_NAME_2021": "abs_mb_category_2021",
        "AREA_ALBERS_SQKM": "abs_mb_area_sqkm_2021",
        "Dwelling": "abs_mb_dwellings_2021",
        "Person": "abs_mb_persons_usual_resident_2021",
    }
).drop(columns="State")

print(f"WA Mesh Blocks available for joining: {len(wa_counts):,}")
display(wa_counts.head())

WA Mesh Blocks available for joining: 43,322


,abs_mb_code_2021,abs_mb_category_2021,abs_mb_area_sqkm_2021,abs_mb_dwellings_2021,abs_mb_persons_usual_resident_2021
0,50000010000,Residential,0.0357,32.0,67.0
1,50000020000,Residential,0.0425,34.0,81.0
2,50000030000,Education,0.0816,0.0,0.0
3,50000040000,Parkland,0.0581,0.0,0.0
4,50000060000,Residential,0.0154,31.0,44.0


## 3. Use the official ABS API to identify the containing MB and SA1

The official 2021 Mesh Block ArcGIS layer is queried once for each site coordinate. The service does not require an API key. Only two attributes are requested:

| API field | Meaning and use |
|---|---|
| `mb_code_2021` | Code for the 2021 Mesh Block intersecting the point. It connects the spatial result to the local count table. |
| `sa1_code_2021` | Code for the 2021 SA1 containing that MB. It is retained as a join key for possible future SA1-level data and is not itself a predictor. |

A separate SA1 request is unnecessary because the MB layer already contains the SA1 code. The function preserves `no_match`, `multiple_matches`, and `request_failed` outcomes instead of silently assigning an uncertain geography. This research-scale run intentionally avoids a production caching or retry system; failed requests remain visible and can be rerun.

In [4]:
session = requests.Session()
session.headers.update({"User-Agent": "CITS5553-capstone-feature-extraction/1.0"})


def query_abs_mesh_block(site_id: str, longitude: float, latitude: float) -> dict:
    """Return the single 2021 ABS Mesh Block and SA1 intersecting a WGS84 point."""
    result = {
        "site_id": site_id,
        "abs_match_status": pd.NA,
        "abs_error_message": pd.NA,
        "abs_mb_code_2021": pd.NA,
        "abs_sa1_code_2021": pd.NA,
    }
    params = {
        "f": "json",
        "where": "1=1",
        "geometry": json.dumps(
            {
                "x": float(longitude),
                "y": float(latitude),
                "spatialReference": {"wkid": 4326},
            }
        ),
        "geometryType": "esriGeometryPoint",
        "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "mb_code_2021,sa1_code_2021",
        "returnGeometry": "false",
    }

    try:
        response = session.get(MB_QUERY_URL, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        if "error" in payload:
            raise RuntimeError(payload["error"].get("message", "ABS API error"))

        matches = payload.get("features", [])
        if len(matches) == 0:
            result["abs_match_status"] = "no_match"
        elif len(matches) > 1:
            result["abs_match_status"] = "multiple_matches"
            result["abs_error_message"] = (
                f"ABS returned {len(matches)} intersecting Mesh Blocks."
            )
        else:
            attributes = matches[0].get("attributes", {})
            mb_code = attributes.get("mb_code_2021")
            sa1_code = attributes.get("sa1_code_2021")
            if mb_code is None or sa1_code is None:
                result["abs_match_status"] = "unexpected_response"
                result["abs_error_message"] = "ABS response omitted MB or SA1 code."
            else:
                result["abs_match_status"] = "matched"
                result["abs_mb_code_2021"] = str(mb_code)
                result["abs_sa1_code_2021"] = str(sa1_code)
    except Exception as error:
        result["abs_match_status"] = "request_failed"
        result["abs_error_message"] = f"{type(error).__name__}: {error}"[:300]

    return result


print(f"Querying the ABS spatial service for {len(sites):,} sites...")
api_matches = pd.DataFrame(
    query_abs_mesh_block(row.site_id, row.longitude, row.latitude)
    for row in sites.itertuples(index=False)
)

display(api_matches.head())
display(api_matches["abs_match_status"].value_counts(dropna=False).to_frame("sites"))

Querying the ABS spatial service for 49 sites...


,site_id,abs_match_status,abs_error_message,abs_mb_code_2021,abs_sa1_code_2021
0,250303-01,matched,<NA>,50391642000,50102101608
1,250303-02,matched,<NA>,50043794000,50102101101
2,250303-03,matched,<NA>,50366540000,50102100909
3,250303-04,matched,<NA>,50196282000,50103101903
4,250303-05,matched,<NA>,50227251000,50201102910


,sites
abs_match_status,
matched,49


## 4. Build and save the source-specific feature table

The API result is joined to the local WA count table through `abs_mb_code_2021`. The final workbook contains one sheet, `ABS_MB_SA1_Features`, and one row per input `site_id`. Its columns are:

| Output column | Interpretation |
|---|---|
| `site_id` | Project primary key used to join this feature table back to `Model_Data`. |
| `abs_match_status`, `abs_error_message` | Spatial/API result and any reason a usable feature row could not be produced. These are QA fields, not predictors. |
| `abs_mb_code_2021`, `abs_sa1_code_2021` | ABS geographic identifiers for traceability and later joins; not continuous predictors. |
| `abs_mb_category_2021` | Dominant category of the containing MB. |
| `abs_mb_area_sqkm_2021` | Official area of the containing MB, not the area of a site-centred buffer. |
| `abs_mb_dwellings_2021`, `abs_mb_persons_usual_resident_2021` | Official raw dwelling and usual-resident counts in the containing MB. |
| `abs_mb_dwellings_per_sqkm_2021`, `abs_mb_persons_per_sqkm_2021` | Raw count divided by official MB area where area is greater than zero. |

The output is written as `.xlsx` because it is a small, reviewable team feature table. Dataset definitions remain in this bilingual notebook rather than being repeated as data columns or as another metadata output. The final code confirms that the row count and unique `site_id` values still match the foundation table.

In [5]:
features = api_matches.merge(
    wa_counts,
    on="abs_mb_code_2021",
    how="left",
    validate="many_to_one",
)

missing_local_counts = (
    features["abs_match_status"].eq("matched")
    & features["abs_mb_category_2021"].isna()
)
features.loc[missing_local_counts, "abs_match_status"] = "matched_no_local_counts"
features.loc[missing_local_counts, "abs_error_message"] = (
    "MB matched by the API but was not found in the local WA counts table."
)

valid_area = features["abs_mb_area_sqkm_2021"].gt(0)
features["abs_mb_dwellings_per_sqkm_2021"] = (
    features["abs_mb_dwellings_2021"]
    .div(features["abs_mb_area_sqkm_2021"])
    .where(valid_area)
)
features["abs_mb_persons_per_sqkm_2021"] = (
    features["abs_mb_persons_usual_resident_2021"]
    .div(features["abs_mb_area_sqkm_2021"])
    .where(valid_area)
)

feature_columns = [
    "site_id",
    "abs_match_status",
    "abs_error_message",
    "abs_mb_code_2021",
    "abs_sa1_code_2021",
    "abs_mb_category_2021",
    "abs_mb_area_sqkm_2021",
    "abs_mb_dwellings_2021",
    "abs_mb_persons_usual_resident_2021",
    "abs_mb_dwellings_per_sqkm_2021",
    "abs_mb_persons_per_sqkm_2021",
]
features = features[feature_columns].copy()

if len(features) != len(sites):
    raise ValueError("Feature output row count no longer matches the model-site input.")
if features["site_id"].duplicated().any():
    raise ValueError("Feature output must contain exactly one row per site_id.")

features.to_excel(
    FEATURE_PATH,
    sheet_name="ABS_MB_SA1_Features",
    index=False,
)

display(features)
display(features["abs_match_status"].value_counts(dropna=False).to_frame("sites"))
print("Saved:", FEATURE_PATH.relative_to(PROJECT_ROOT))

,site_id,abs_match_status,abs_error_message,abs_mb_code_2021,abs_sa1_code_2021,abs_mb_category_2021,abs_mb_area_sqkm_2021,abs_mb_dwellings_2021,abs_mb_persons_usual_resident_2021,abs_mb_dwellings_per_sqkm_2021,abs_mb_persons_per_sqkm_2021
0,250303-01,matched,<NA>,50391642000,50102101608,Residential,7.9418,35.0,74.0,4.407061,9.317787
1,250303-02,matched,<NA>,50043794000,50102101101,Industrial,0.3167,0.0,0.0,0.000000,0.000000
2,250303-03,matched,<NA>,50366540000,50102100909,Transport,0.1362,0.0,0.0,0.000000,0.000000
3,250303-04,matched,<NA>,50196282000,50103101903,Commercial,0.0568,0.0,0.0,0.000000,0.000000
4,250303-05,matched,<NA>,50227251000,50201102910,Commercial,0.0616,0.0,0.0,0.000000,0.000000
5,250303-06,matched,<NA>,50143360000,50102101411,Residential,0.0455,36.0,83.0,791.208791,1824.175824
6,250303-07,matched,<NA>,50099181000,50103101805,Commercial,0.0779,0.0,0.0,0.000000,0.000000
7,250303-08,matched,<NA>,50062942000,50102100704,Residential,0.1137,12.0,27.0,105.540897,237.467018
8,250303-09,matched,<NA>,50333692000,50102101602,Parkland,0.0938,0.0,0.0,0.000000,0.000000
9,250303-10,matched,<NA>,50033911000,50103101712,Commercial,0.0325,0.0,0.0,0.000000,0.000000


,sites
abs_match_status,
matched,49


Saved: outputs/02_imagery_features/abs-mesh-block-sa1/haoqi-01-abs-mb-sa1-features.xlsx
